# 作业2：列表，字典和类


In [1]:
import os
import uuid
import hashlib
import platform

# 修改为自己的学号
STUDENT_ID = 20244081104

CONDA_ENV = os.environ.get("CONDA_DEFAULT_ENV", "base")
print(f"[Conda 环境] {CONDA_ENV}")

MACHINE_FP = hashlib.sha256(
    f"{platform.node()}|{platform.machine()}|{platform.system()}|{uuid.getnode()}".encode()
).hexdigest()[:16]
RUN_ID = f"{STUDENT_ID}-{MACHINE_FP}"
print(f"[学号]       {STUDENT_ID}")
print(f"[机器指纹] {MACHINE_FP}")
print(f"[RUN_ID]   {RUN_ID}")


[Conda 环境] pytorch
[学号]       20244081104
[机器指纹] 90bdc247d63f884e
[RUN_ID]   20244081104-90bdc247d63f884e


## 字符串的乐趣


### 1. is_palindrome


In [2]:
def is_palindrome(s):
    """回文判断：忽略空格和大小写。题目保证输入只含英文字母、数字 0-9 和空格。"""
    cleaned = [ch.lower() for ch in s if ch != " "]
    return cleaned == cleaned[::-1]


for t in ["level", "kayak", "pop", "tacocat", "T A C O cat",
          "rats live on no evil star", "Was it a car or a cat I saw?", "goat"]:
    print(f"{t!r:40s} -> {is_palindrome(t)}")


'level'                                  -> True
'kayak'                                  -> True
'pop'                                    -> True
'tacocat'                                -> True
'T A C O cat'                            -> True
'rats live on no evil star'              -> True
'Was it a car or a cat I saw?'           -> False
'goat'                                   -> False


题目说输入只由英文字母、数字和空格组成，所以只需要去掉空格并统一小写，不用处理标点。这样 `T A C O cat`、`rats live on no evil star` 都能正确判为回文。注意 `Was it a car or a cat I saw?` 里的问号不在题目声明的字符集内，严格按题目假设它不会出现；如果要在真实文本上用，得先把标点也过滤掉。


### 2. is_abecedarian


In [3]:
def is_abecedarian(s):
    """字母是否按字母顺序排列（允许重复，忽略空格）。"""
    letters = [ch.lower() for ch in s if ch != " "]
    for i in range(1, len(letters)):
        if letters[i] < letters[i - 1]:
            return False
    return True


for t in ["adder", "beet", "dog", "cat", "abcd efgh xyz", "aabbcc", ""]:
    print(f"{t!r:16s} -> {is_abecedarian(t)}")


'adder'          -> True
'beet'           -> True
'dog'            -> False
'cat'            -> False
'abcd efgh xyz'  -> True
'aabbcc'         -> True
''               -> True


判断标准是"后一个字母不能小于前一个"，相等是允许的，所以重复字母（`beet`、`aabbcc`）仍然返回 True。这里也不能先 `sorted(s)` 再比：空格在 ASCII 里比所有字母都小，`abcd efgh xyz` 带着空格排序后会变样，误判成 False；必须先把空格去掉再比较。


### 3. remove_vowels


In [4]:
def remove_vowels(s):
    """去掉所有元音 a、e、i、o、u。"""
    vowels = "aeiou"
    return "".join(ch for ch in s if ch.lower() not in vowels)


for t in ["cat", "audio", "aeiou", "goat", "Python", "", "A E I O U"]:
    print(f"{t!r:12s} -> {remove_vowels(t)!r}")


'cat'        -> 'ct'
'audio'      -> 'd'
'aeiou'      -> ''
'goat'       -> 'gt'
'Python'     -> 'Pythn'
''           -> ''
'A E I O U'  -> '    '


元音只有 a、e、i、o、u 五个，`y` 不算。全部由元音组成的字符串要正确返回空串；大小写都要处理，所以先 `ch.lower()` 再判断。`'A E I O U'` 里的空格不是元音，会被保留下来，结果是 4 个空格。


## 列表的乐趣


### 1. list_reverse


In [5]:
def list_reverse(lst):
    """返回反转后的新列表；输入不是列表时抛 TypeError。"""
    if type(lst) is not list:
        raise TypeError(f"参数必须是 list，收到的是 {type(lst).__name__}")
    return lst[::-1]


print(list_reverse([1, 2, 3]))
print(list_reverse([]))
print(list_reverse(["a"]))

original = [1, 2, 3]
result = list_reverse(original)
print(f"\n原列表未被修改: {original}")
print(f"返回的是新对象: {result is not original}")

for bad in ("abc", (1, 2), 123, None):
    try:
        list_reverse(bad)
    except TypeError as e:
        print(f"{bad!r:10s} -> TypeError: {e}")


[3, 2, 1]
[]
['a']

原列表未被修改: [1, 2, 3]
返回的是新对象: True
'abc'      -> TypeError: 参数必须是 list，收到的是 str
(1, 2)     -> TypeError: 参数必须是 list，收到的是 tuple
123        -> TypeError: 参数必须是 list，收到的是 int
None       -> TypeError: 参数必须是 list，收到的是 NoneType


这一小节题目要求做错误检查。用切片 `lst[::-1]` 而不是 `lst.reverse()`：后者是原地反转、返回 `None`，会既改掉调用方传进来的列表、又拿不到结果。题目说"输入不是列表应引发适当的错误"，这里用 `type(lst) is not list` 严格判断，元组和字符串也会被拒。


### 2. is_sorted


In [6]:
def is_sorted(seq):
    """判断序列是否按升序排列。只遍历一次，不做类型检查。"""
    for i in range(len(seq) - 1):
        if seq[i] > seq[i + 1]:
            return False
    return True


cases = [([1, 2, 3], True), ([1, 1, 2], True), ([3, 2, 1], False),
         ([], True), ([5], True), ("abc", True), ("cba", False), ((1, 2, 2, 3), True)]
for seq, expected in cases:
    got = is_sorted(seq)
    print(f"{seq!r:18s} -> {got!s:5s} (期望 {expected})  {'OK' if got == expected else 'MISMATCH'}")


[1, 2, 3]          -> True  (期望 True)  OK
[1, 1, 2]          -> True  (期望 True)  OK
[3, 2, 1]          -> False (期望 False)  OK
[]                 -> True  (期望 True)  OK
[5]                -> True  (期望 True)  OK
'abc'              -> True  (期望 True)  OK
'cba'              -> False (期望 False)  OK
(1, 2, 2, 3)       -> True  (期望 True)  OK


题目要求只遍历一次，所以不能用 `seq == sorted(seq)`（sorted 会先整体排序、再整体比较，等于遍历了不止一遍，而且额外占 O(n) 空间）。这里用相邻两项比较，只要出现前一项大于后一项就立刻返回 False，最好情况 O(1) 就能退出。

关于 Python 动态类型：这个函数完全没检查 `seq` 是什么类型，只要它支持索引和 `len()`、元素之间能用 `>` 比较就能工作，所以列表、字符串、元组都能直接传进来（上面的测试里就同时用了三种）。这就是"鸭子类型"——不关心对象是哪一类，只关心它能不能做这几个操作。`[]` 和单元素序列没有相邻可比，自然返回 True。


## 作为向量的元组


### 1. vec_scalar_mult


In [7]:
def vec_scalar_mult(t, s):
    """元组 t 的每个元素乘以标量 s，返回等长元组；类型不对抛 TypeError。"""
    if not isinstance(t, tuple):
        raise TypeError(f"第一个参数必须是 tuple，收到的是 {type(t).__name__}")
    if not isinstance(s, (int, float)) or isinstance(s, bool):
        raise TypeError(f"标量必须是 int 或 float，收到的是 {type(s).__name__}")
    for i in range(len(t)):
        if not isinstance(t[i], (int, float)) or isinstance(t[i], bool):
            raise TypeError(f"第 {i} 个元素不是数字：{t[i]!r}")
    return tuple(x * s for x in t)


print(vec_scalar_mult((1, 2, 3), 2))
print(vec_scalar_mult((1, 2, 3), 0.5))
print(vec_scalar_mult((1.5, -2.0), 3))
print(vec_scalar_mult((), 5))

for bad in [([1, 2], 2), ((1, 2), "x"), ((1, "a"), 2)]:
    try:
        vec_scalar_mult(*bad)
    except TypeError as e:
        print(f"{bad!r:18s} -> TypeError: {e}")


(2, 4, 6)
(0.5, 1.0, 1.5)
(4.5, -6.0)
()
([1, 2], 2)        -> TypeError: 第一个参数必须是 tuple，收到的是 list
((1, 2), 'x')      -> TypeError: 标量必须是 int 或 float，收到的是 str
((1, 'a'), 2)      -> TypeError: 第 1 个元素不是数字：'a'


题目明确要求这里做类型检查，错误消息要能看懂。注意两个细节：一是 `bool` 在 Python 里是 `int` 的子类，`isinstance(True, int)` 为 True，如果不单独排除，`vec_scalar_mult((1,2), True)` 会被当成乘以 1 而悄悄通过；二是返回空元组 `()` 时也要正常工作，不能因为"没有元素"就跳过返回。


### 2. vec_inner_product


In [8]:
def vec_inner_product(u, v):
    """两个等长向量的内积，返回浮点数。"""
    if not isinstance(u, tuple) or not isinstance(v, tuple):
        raise TypeError("两个参数都必须是 tuple")
    if len(u) != len(v):
        raise ValueError(f"长度不一致：{len(u)} vs {len(v)}")

    total = 0.0
    for j in range(len(u)):
        total += u[j] * v[j]
    return total


print(vec_inner_product((1, 2, 3), (4, 5, 6)))
print(vec_inner_product((1, 0), (0, 1)))
print(vec_inner_product((), ()))

try:
    vec_inner_product([1, 2], (3, 4))
except TypeError as e:
    print("TypeError:", e)

try:
    vec_inner_product((1, 2), (1, 2, 3))
except ValueError as e:
    print("ValueError:", e)


32.0
0.0
0.0
TypeError: 两个参数都必须是 tuple
ValueError: 长度不一致：2 vs 3


内积按定义 `Σ xⱼyⱼ` 逐项相乘再累加，`total` 初始化为 `0.0` 保证返回的是浮点数（即便两个参数都是整数元组）。两类错误要区分开：类型不对抛 TypeError，长度不一致抛 ValueError，这是 Python 的惯例——类型错了是调用方式的问题，值错了是数据的问题。


### 3. check_valid_mx


In [9]:
def check_valid_mx(mx):
    """元组的元组表示的矩阵是否有效：每行都是元组、行等长、元素都是数字。"""
    if not isinstance(mx, tuple):
        return False
    if len(mx) == 0:
        return True                          # 空矩阵按有效处理
    if not all(isinstance(row, tuple) for row in mx):
        return False
    n = len(mx[0])
    for row in mx:
        if len(row) != n:
            return False
    for row in mx:
        for x in row:
            if not isinstance(x, (int, float)) or isinstance(x, bool):
                return False
    return True


cases = [(((1, 2), (3, 4)), True),
         (((1, 2, 3), (4, 5, 6)), True),
         (((1.5, 2), (3, 4.5)), True),
         (((1, 2), (3,)), False),
         (((1, 2), [3, 4]), False),
         (([1, 2], [3, 4]), False),
         (((1, "a"), (3, 4)), False),
         (((1, 2),), True),
         ((), True)]
for mx, expected in cases:
    got = check_valid_mx(mx)
    print(f"{str(mx):28s} -> {got!s:5s} (期望 {expected})  {'OK' if got == expected else 'MISMATCH'}")


((1, 2), (3, 4))             -> True  (期望 True)  OK
((1, 2, 3), (4, 5, 6))       -> True  (期望 True)  OK
((1.5, 2), (3, 4.5))         -> True  (期望 True)  OK
((1, 2), (3,))               -> False (期望 False)  OK
((1, 2), [3, 4])             -> False (期望 False)  OK
([1, 2], [3, 4])             -> False (期望 False)  OK
((1, 'a'), (3, 4))           -> False (期望 False)  OK
((1, 2),)                    -> True  (期望 True)  OK
()                           -> True  (期望 True)  OK


三个条件（外层每个元素是元组、内层元组等长、元素都是 int 或 float）用三次判断分别落实，只要有一项不满足就返回 False，不能抛异常——这个函数的职责是判断，把决定权交给调用方。边界情况 `()`（0×0 矩阵）我按有效处理：它满足"每一行都是元组、所有行等长、元素都是数字"这三条，属于空真。


### 4. mx_vec_mult


In [10]:
def mx_vec_mult(mx, vec):
    """矩阵乘向量。

    题目原文是"返回 向量 与 矩阵 相乘 得到的数值元组"，但函数名 mx_vec_mult
    以及题目里矩阵在前、向量在后的参数顺序都指向 M·v，这里按 M·v 实现：
    结果第 i 个分量是矩阵第 i 行与向量的内积，长度等于矩阵行数。
    """
    if not check_valid_mx(mx):
        raise TypeError(f"第一个参数不是有效矩阵：{mx!r}")
    if not isinstance(vec, tuple):
        raise TypeError(f"第二个参数必须是 tuple，收到的是 {type(vec).__name__}")
    for i in range(len(vec)):
        if not isinstance(vec[i], (int, float)) or isinstance(vec[i], bool):
            raise TypeError(f"向量第 {i} 个元素不是数字：{vec[i]!r}")
    if len(mx) == 0:
        raise ValueError("矩阵为空，无法相乘")

    n = len(mx[0])
    if len(vec) != n:
        raise ValueError(f"维度不匹配：矩阵有 {n} 列，向量是 {len(vec)} 维")

    result = []
    for row in mx:
        s = 0
        for j in range(n):
            s += row[j] * vec[j]
        result.append(s)
    return tuple(result)


M = ((1, 2, 3), (4, 5, 6))
v3 = (1, 1, 1)
print("M =", M, "(2x3)")
print("v =", v3, "(3 维)")
print("结果 =", mx_vec_mult(M, v3), "(2 维)")

print("\n单位矩阵乘任意向量:")
print(mx_vec_mult(((1, 0), (0, 1)), (7, 9)))

print("\n异常测试:")
for args in [(((1, 2), (3, 4)), (1, 2, 3)), (((1, 2),), [1]), ("bad", (1,))]:
    try:
        mx_vec_mult(*args)
    except (TypeError, ValueError) as e:
        print(f"{type(e).__name__}: {e}")


M = ((1, 2, 3), (4, 5, 6)) (2x3)
v = (1, 1, 1) (3 维)
结果 = (6, 15) (2 维)

单位矩阵乘任意向量:
(7, 9)

异常测试:
ValueError: 维度不匹配：矩阵有 2 列，向量是 3 维
TypeError: 第二个参数必须是 tuple，收到的是 list
TypeError: 第一个参数不是有效矩阵：'bad'


题目要求"`check_valid_mx` 应检查参数是否合理，对于不合理的输入应引发适当的错误"，所以这里第一关就是调 `check_valid_mx(mx)`，不合法直接抛 TypeError。之后还要单独检查向量：维度必须等于矩阵的列数，否则抛 ValueError。`((1,2,3),(4,5,6)) · (1,1,1)` 得到 `(6, 15)`，长度 2 等于矩阵行数，符合 M·v 的规则。


## 更多向量的乐趣


### 1. Vector 类


In [11]:
def check_coords(coords, dim):
    """把 coords 规整成元组并校验；不提供时返回 dim 个 0。"""
    if coords is None:
        return (0,) * dim
    if not isinstance(coords, (list, tuple)):
        raise TypeError(f"元素必须是 list 或 tuple，收到的是 {type(coords).__name__}")
    if len(coords) != dim:
        raise ValueError(f"维度 {dim} 与元素个数 {len(coords)} 不一致")
    for x in coords:
        if not isinstance(x, (int, float)) or isinstance(x, bool):
            raise TypeError(f"元素必须都是整数或浮点数，收到 {x!r}")
    return tuple(coords)


class Vector:
    """d 维向量。第一个参数是维度，第二个参数是元素列表或元组（可省略）。"""

    def __init__(self, dim, coords=None):
        if not isinstance(dim, int) or isinstance(dim, bool):
            raise TypeError(f"维度必须是整数，收到的是 {type(dim).__name__}")
        if dim < 0:
            raise ValueError(f"维度不能为负：{dim}")

        self.dim = dim
        self.coords = check_coords(coords, dim)

    def __repr__(self):
        return f"Vector({self.dim}, {self.coords})"


print(Vector(2, [2, 3]))
print(Vector(3, (1.5, -2, 0)))
print(Vector(4))
print(Vector(0, []))

for args in [(2, [1, 2, 3]), (-1, []), (2.5, [1, 2]), (2, "ab"), (2, [1, "a"])]:
    try:
        Vector(*args)
    except (TypeError, ValueError) as e:
        print(f"{args!r:16s} -> {type(e).__name__}: {e}")


Vector(2, (2, 3))
Vector(3, (1.5, -2, 0))
Vector(4, (0, 0, 0, 0))
Vector(0, ())
(2, [1, 2, 3])   -> ValueError: 维度 2 与元素个数 3 不一致
(-1, [])         -> ValueError: 维度不能为负：-1
(2.5, [1, 2])    -> TypeError: 维度必须是整数，收到的是 float
(2, 'ab')        -> TypeError: 元素必须是 list 或 tuple，收到的是 str
(2, [1, 'a'])    -> TypeError: 元素必须都是整数或浮点数，收到 'a'


默认行为的选择：题目说"如果只提供维度而不提供元素，请选择合理的默认行为"，我选全零向量——数学上零向量在每一维都存在，是最中性的初值，不像随机数那样给对象平添一层假设。

`coords` 统一用 `check_coords` 转成元组再存，有两个原因：一是元组不可变，外部拿到之后没法原地改掉、把 `dim` 和实际长度弄脱节；二是长度校验只写一遍，后面 `Vector.dot`、乘法运算符返回新对象时都能直接复用。

这一问只定义"存数据"的部分。题目接下来的两问（比较运算符、`dot`、乘法运算符）写的还是这同一个 `Vector` 类，我在对应的小节里给这个类补方法，而不是另起一个新类。


### 2. 维度和元素应是类属性还是实例属性？


答：应该是实例属性。

类属性写在类体里、由这个类的所有实例共享，一份内存大家公用；实例属性挂在 `self` 上，每个对象各存一份。

如果 `dim` 和 `coords` 写成类属性，那么 `a = Vector(2, [1, 2])` 和 `b = Vector(3, [1, 2, 3])` 先后创建时，两次 `__init__` 写的都是同一个类变量，后建的会把先建的覆盖掉——`a.dim` 会莫名其妙变成 3，所有向量都塌成"最后一个"的样子，根本无法表示多个不同维度、不同坐标的向量同时存在。

而向量的维度和坐标正是每个对象自己的数据：a 是 2 维 (1, 2)，b 是 3 维 (1, 2, 3)，它们必须互不影响，所以只能挂在实例上（`self.dim`、`self.coords`），随对象创建而存在、随对象销毁而消失。

一般规律是：类属性描述"这一类事物共有的特征"或所有实例共享的常量（比如默认容差、计数器上限），实例属性描述"这一个对象的具体取值"。上面代码里 `Vector(2, [2, 3])` 和 `Vector(4)` 能各自保持自己的维度同时存在，就是实例属性的直接证据。


### 3. 比较运算符


In [12]:
def _same_dim(self, other):
    """大小比较前检查：必须是 Vector，且维度相同。"""
    if not isinstance(other, Vector):
        raise TypeError(f"只能和 Vector 比较，收到的是 {type(other).__name__}")
    if self.dim != other.dim:
        raise ValueError(f"维度不同无法比较：{self.dim} vs {other.dim}")


def vector_eq(self, other):
    if not isinstance(other, Vector):
        return NotImplemented
    if self.dim != other.dim:
        raise ValueError(f"维度不同无法比较：{self.dim} vs {other.dim}")
    return self.coords == other.coords


def vector_lt(self, other):
    _same_dim(self, other)
    return self.coords < other.coords


def vector_le(self, other):
    _same_dim(self, other)
    return self.coords <= other.coords


def vector_gt(self, other):
    _same_dim(self, other)
    return self.coords > other.coords


def vector_ge(self, other):
    _same_dim(self, other)
    return self.coords >= other.coords


Vector.__eq__ = vector_eq
Vector.__lt__ = vector_lt
Vector.__le__ = vector_le
Vector.__gt__ = vector_gt
Vector.__ge__ = vector_ge

a = Vector(2, [2, 4])
b = Vector(2, [2, 5])
c = Vector(2, [2, 4])

print(f"a = {a}")
print(f"b = {b}")
print(f"a == c : {a == c}")
print(f"a != b : {a != b}")
print(f"a <  b : {a < b}")
print(f"a <= c : {a <= c}")
print(f"b >  a : {b > a}")
print(f"b >= a : {b >= a}")

print("\n维度不同时:")
try:
    print(a < Vector(3, [2, 4, 0]))
except ValueError as e:
    print("ValueError:", e)

print("\n和非 Vector 比较:")
try:
    print(a < 5)
except TypeError as e:
    print("TypeError:", e)

print("a == 5 ->", a == 5)


a = Vector(2, (2, 4))
b = Vector(2, (2, 5))
a == c : True
a != b : True
a <  b : True
a <= c : True
b >  a : True
b >= a : True

维度不同时:
ValueError: 维度不同无法比较：2 vs 3

和非 Vector 比较:
TypeError: 只能和 Vector 比较，收到的是 int
a == 5 -> False


题目的比较规则和 Python 元组的字典序完全一致（先比第一个坐标，相等再比第二个），而 `self.coords` 本来就是元组，所以直接把两个元组比较就行，不用自己写逐项循环。

`__eq__` 和另外四个要分开处理：`==` 遇到不是 Vector 的对象返回 `NotImplemented`，让 Python 去试对方的反运算，最终得到 `False`，而不是无缘无故抛异常；`<`、`<=`、`>`、`>=` 遇到非 Vector 就抛 `TypeError`，因为"向量和数字比大小"本身没有定义。`!=` 不用单独实现，Python 3 会自动拿 `__eq__` 的结果取反。

全部比较运算符都直接复用元组的字典序比较，这是我选择用元组存 `coords` 带来的便利。

题目的第 3、4、5 问是在同一个 `Vector` 类上继续加能力，所以这里以 `Vector.__eq__ = vector_eq` 的形式把方法挂回已有的类——和直接写在 `class` 体内效果完全一样，把这几问拆成独立单元格反而是为了让每一步的结果更好验证。


### 4. Vector.dot


In [13]:
def vector_dot(self, other):
    if not isinstance(other, Vector):
        raise TypeError(f"参数必须是 Vector，收到的是 {type(other).__name__}")
    if self.dim != other.dim:
        raise ValueError(f"维度不一致：{self.dim} vs {other.dim}")

    total = 0.0
    for i in range(self.dim):
        total += self.coords[i] * other.coords[i]
    return total


Vector.dot = vector_dot

u = Vector(3, [1, 2, 3])
w = Vector(3, [4, 5, 6])
print(f"u = {u}")
print(f"w = {w}")
print(f"u.dot(w) = {u.dot(w)}")
print(f"w.dot(u) = {w.dot(u)}")
print(f"u.dot(u) = {u.dot(u)}")

try:
    u.dot((1, 2, 3))
except TypeError as e:
    print("TypeError:", e)

try:
    u.dot(Vector(2, [1, 2]))
except ValueError as e:
    print("ValueError:", e)


u = Vector(3, (1, 2, 3))
w = Vector(3, (4, 5, 6))
u.dot(w) = 32.0
w.dot(u) = 32.0
u.dot(u) = 14.0
TypeError: 参数必须是 Vector，收到的是 tuple
ValueError: 维度不一致：3 vs 2


和 `vec_inner_product` 是同一个计算，只是换成对象方法的形式，`total` 初始化为 `0.0` 保证返回浮点数。内积是对称的，`u.dot(w)` 与 `w.dot(u)` 都是 32.0；`u.dot(u) = 14.0` 是自身内积，等于模长的平方。异常同样分两类：参数不是 Vector 抛 `TypeError`，维度不一致抛 `ValueError`。


### 5. Vector 的乘法运算符


In [14]:
def vector_mul(self, other):
    """标量乘法与逐元素乘法共用同一个 * 运算符。"""
    if isinstance(other, bool):
        raise TypeError("不能和 bool 相乘")
    if isinstance(other, (int, float)):
        return Vector(self.dim, [x * other for x in self.coords])
    if isinstance(other, Vector):
        if self.dim != other.dim:
            raise ValueError(f"维度不一致：{self.dim} vs {other.dim}")
        return Vector(self.dim, [x * y for x, y in zip(self.coords, other.coords)])
    return NotImplemented


Vector.__mul__ = vector_mul
Vector.__rmul__ = vector_mul          # 让 2 * v 也能工作

v = Vector(2, [2, 3])
w2 = Vector(2, [4, 5])

print(f"v       = {v}")
print(f"v * 2   = {v * 2}")
print(f"2 * v   = {2 * v}")
print(f"v * 2.5 = {v * 2.5}")
print(f"v * w2  = {v * w2}")
print(f"v * v   = {v * v}")

print("\n左乘右乘结果一致:", (v * 2).coords == (2 * v).coords)

try:
    print(v * Vector(3, [1, 2, 3]))
except ValueError as e:
    print("ValueError:", e)

try:
    print(v * "x")
except TypeError as e:
    print("TypeError:", e)


v       = Vector(2, (2, 3))


v * 2   = Vector(2, (4, 6))
2 * v   = Vector(2, (4, 6))
v * 2.5 = Vector(2, (5.0, 7.5))
v * w2  = Vector(2, (8, 15))
v * v   = Vector(2, (4, 9))

左乘右乘结果一致: True
ValueError: 维度不一致：2 vs 3
TypeError: can't multiply sequence by non-int of type 'Vector'


`v * 2` 和 `2 * v` 都能工作，靠的是把同一个函数挂到了 `__mul__` 和 `__rmul__` 上：`v * 2` 里左边是 Vector，直接调用 `v.__mul__(2)`；`2 * v` 里左边是 int，int 不认识 Vector，于是 Python 反过来调用右边对象的 `v.__rmul__(2)`。只实现 `__mul__` 的话，`2 * v` 会报 `TypeError: unsupported operand type(s) for *: 'int' and 'Vector'`。

两个 Vector 相乘是逐元素相乘（和 NumPy 的 `*` 一样，不是内积），题目例子 `Vector(2,(2,3)) * Vector(2,(4,5))` 结果就是 `Vector(2, (8, 15))`。

两端都不是 Vector、也都不是数字时（比如 `v * "x"`），函数返回 `NotImplemented`，Python 于是再去试字符串的 `__rmul__`，最后抛出解释器自己的错误消息——这正是 `NotImplemented` 的用途：告诉 Python"我处理不了，你接着找别人"，而不是自作主张抛一个自己编的异常。
